# LSTM Encoder

In this notebook we build the encoder half of the VAE.

The encoder takes a full sketch -- a sequence of `(dx, dy, pen_lifted)` steps -- and compresses it into two vectors: `mu` and `log_var`. These define a probability distribution in latent space. We then sample a vector `z` from that distribution using the reparameterisation trick.

This `z` is the compressed representation of the sketch. The decoder (next notebook) will try to reconstruct the original sketch from `z` alone.

## 0. Setup

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import json
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader

# Reuse the data pipeline from the previous notebooks
def drawing_to_stroke3(drawing):
    strokes = []
    for stroke in drawing:
        xs, ys = stroke[0], stroke[1]
        for i in range(len(xs)):
            dx = xs[i] - xs[i-1] if i > 0 else 0
            dy = ys[i] - ys[i-1] if i > 0 else 0
            pen_lifted = 1 if i == len(xs) - 1 else 0
            strokes.append([dx, dy, pen_lifted])
    return np.array(strokes, dtype=np.float32)

def normalise_stroke3(stroke3):
    s = stroke3.copy()
    coords = s[:, :2]
    s[:, :2] = (coords - coords.mean(axis=0)) / (coords.std(axis=0) + 1e-8)
    return s

class QuickDrawDataset(Dataset):
    def __init__(self, path, max_len=200, max_samples=5000):
        self.samples = []
        with open(path) as f:
            for i, line in enumerate(f):
                if i >= max_samples: break
                d  = json.loads(line)
                s3 = drawing_to_stroke3(d['drawing'])
                s3 = normalise_stroke3(s3)
                if len(s3) > max_len: s3 = s3[:max_len]
                self.samples.append(torch.tensor(s3, dtype=torch.float32))
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx): return self.samples[idx]

print('Setup complete.')


## 1. Padding variable-length sequences

Sequences in a batch can have different lengths. To stack them into a single tensor we pad shorter sequences with zeros up to the length of the longest one in the batch.

PyTorch has built-in utilities for this: `pad_sequence` and `pack_padded_sequence`.

In [ ]:
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence

def collate_fn(batch):
    '''
    Custom collate function for DataLoader.
    Pads sequences in a batch to the same length.

    Returns:
        padded  : tensor of shape (batch, max_seq_len, 3)
        lengths : list of original sequence lengths
    '''
    lengths = [seq.shape[0] for seq in batch]
    padded  = pad_sequence(batch, batch_first=True, padding_value=0.0)
    return padded, lengths

# Test it
import urllib.request
os_import = __import__('os')
os_import.makedirs('data', exist_ok=True)
url  = 'https://storage.googleapis.com/quickdraw_dataset/full/simplified/cat.ndjson'
path = 'data/cat.ndjson'
if not os_import.path.exists(path):
    urllib.request.urlretrieve(url, path)

dataset = QuickDrawDataset('data/cat.ndjson', max_len=200, max_samples=2000)
loader  = DataLoader(dataset, batch_size=4, shuffle=True, collate_fn=collate_fn)

padded, lengths = next(iter(loader))
print('Padded batch shape:', padded.shape)   # (4, max_len_in_batch, 3)
print('Sequence lengths  :', lengths)


In [ ]:
# TODO: What does padding_value=0.0 mean for a stroke sequence?
#       Is a zero vector (dx=0, dy=0, pen_lifted=0) a meaningful stroke?
#       Write your answer as a comment.
# YOUR CODE HERE

# A padding value of 0.0 means dx=0, dy=0, and pen_lifted=0.
# This represents holding the pen completely still on the paper without moving.
# It is not a meaningful stroke for drawing a shape, which makes it an ideal dummy value for padding since it doesn't add new lines.


## 2. The LSTM Encoder

The encoder is a bidirectional LSTM. It reads the stroke sequence in both directions (forward and backward) and summarises the entire sequence into a single hidden state vector.

We then pass that hidden state through two linear layers to produce `mu` and `log_var` -- the mean and log-variance of the latent distribution.

In [ ]:
class Encoder(nn.Module):
    def __init__(self, input_dim=3, hidden_dim=256, latent_dim=128, num_layers=1):
        '''
        input_dim  : number of features per step (3 for stroke-3)
        hidden_dim : size of the LSTM hidden state
        latent_dim : size of the latent vector z
        num_layers : number of stacked LSTM layers
        '''
        super().__init__()
        self.hidden_dim = hidden_dim
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True   # reads sequence forward AND backward
        )
        # bidirectional doubles the output size
        self.fc_mu     = nn.Linear(hidden_dim * 2, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim * 2, latent_dim)

    def forward(self, x, lengths):
        '''
        x       : padded tensor shape (batch, seq_len, 3)
        lengths : list of actual sequence lengths per sample

        Returns:
            mu     : shape (batch, latent_dim)
            logvar : shape (batch, latent_dim)
        '''
        # Pack so the LSTM ignores padding positions
        packed = pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)
        _, (h, _) = self.lstm(packed)

        # h shape: (num_directions * num_layers, batch, hidden_dim)
        # Concatenate the forward and backward final hidden states
        h = torch.cat([h[0], h[1]], dim=-1)   # shape: (batch, hidden_dim * 2)

        mu     = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar

encoder = Encoder(input_dim=3, hidden_dim=256, latent_dim=128)
print(encoder)
print()
total_params = sum(p.numel() for p in encoder.parameters())
print(f'Total parameters: {total_params:,}')


## 3. The Reparameterisation Trick

We want to sample `z` from the distribution defined by `mu` and `log_var`. But sampling is not differentiable -- you cannot backpropagate through a random operation.

The reparameterisation trick fixes this: instead of sampling `z` directly, we sample a noise vector `epsilon` from a standard normal distribution and compute `z = mu + epsilon * std`. The randomness is now in `epsilon`, which is not a learned parameter, so gradients flow through `mu` and `std` cleanly.

In [ ]:
def reparameterise(mu, logvar):
    '''
    Sample z from N(mu, std) using the reparameterisation trick.
    z = mu + epsilon * std,  where epsilon ~ N(0, 1)

    During inference (eval mode) we can just return mu directly.
    '''
    std     = torch.exp(0.5 * logvar)   # std = exp(log_var / 2)
    epsilon = torch.randn_like(std)     # sample noise, same shape as std
    z       = mu + epsilon * std
    return z

# Test with a batch
padded, lengths = next(iter(loader))
mu, logvar = encoder(padded, lengths)
z = reparameterise(mu, logvar)

print('mu shape    :', mu.shape)     # (batch, latent_dim)
print('logvar shape:', logvar.shape)
print('z shape     :', z.shape)


In [ ]:
# TODO: What happens to z if we set logvar to a very large negative number (e.g. -10)?
#       What does this mean geometrically in latent space?
#       Hint: what is exp(0.5 * -10)?
# YOUR CODE HERE
# If logvar is a large negative number like -10, the std becomes exp(-5), which is practically 0.
# This means the random noise term (epsilon * std) disappears, and z becomes exactly equal to mu.
# Geometrically, the probability distribution collapses into a single, deterministic point in the latent space rather than a spread-out region.


## 4. The KL Divergence loss

The VAE has two loss terms:
1. **Reconstruction loss** -- how well does the decoder recreate the original sketch from z?
2. **KL divergence** -- how close is the learned distribution N(mu, std) to a standard normal N(0, 1)?

The KL term regularises the latent space. Without it, the encoder could learn to map every sketch to a tiny, distinct region of latent space -- which makes interpolation and generation impossible.

The closed-form KL divergence between N(mu, std) and N(0, 1) is:

In [ ]:
def kl_loss(mu, logvar):
    '''
    KL divergence between N(mu, std) and N(0, 1).
    Closed-form: -0.5 * sum(1 + log_var - mu^2 - exp(log_var))
    We take the mean over the batch.
    '''
    kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    return kl

# Test
mu_test     = torch.zeros(4, 128)   # perfect latent: mu=0, logvar=0 means N(0,1)
logvar_test = torch.zeros(4, 128)
print('KL loss when mu=0, logvar=0 (should be ~0):', kl_loss(mu_test, logvar_test).item())

mu_bad     = torch.ones(4, 128) * 5   # latent far from N(0,1)
logvar_bad = torch.zeros(4, 128)
print('KL loss when mu=5, logvar=0 (should be large):', kl_loss(mu_bad, logvar_bad).item())


In [ ]:
# TODO: What does a KL loss of 0 mean geometrically?
#       Why do we want the latent space to be close to N(0, 1) specifically?
# YOUR CODE HERE
# A KL loss of 0 means the learned distribution perfectly matches the standard normal distribution N(0, 1).
# We want the latent space to be close to N(0, 1) so it remains continuous, dense, and centered around the origin.
# Without this, the model would map sketches to isolated, disjoint points, making it impossible to smoothly interpolate between drawings or generate new ones from random noise.


## Done!

You have built the encoder half of the VAE. It takes a stroke sequence and produces a latent vector `z`.
Next: `vae.ipynb` -- the decoder and full training loop.